# Dynamic Baseline GDP Recovery Calculation

This notebook creates a country-level GDP recovery summary for two shock periods:

- **2007 shock window**
- **2019 shock window**

The final output is:

| Country | Recovery_2007 | Recovery_2019 |
|---|---:|---:|

## Core Idea

The input GDP values are **year-on-year GDP growth percentages**, not absolute GDP levels.

So a positive GDP growth rate after a negative year does **not automatically mean recovery**.

Example:

```text
Baseline GDP index = 100
Year 1 growth = -5%  → GDP index = 95
Year 2 growth = +3%  → GDP index = 97.85

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------

input_file = Path("Data/Raw/Raw_Files/OECD_GDP_Growth_2000_2025.csv")

output_dir = Path("Data/Processed/Recovery")
output_dir.mkdir(parents=True, exist_ok=True)

# Input file columns
country_col = "Country"
year_col = "Year"
value_col = "Value"

# Shock anchor years
# These are NOT fixed baselines.
# They define the shock windows we inspect.
shock_anchor_years = [2007, 2019]

# Check two years after each anchor:
# 2007 -> 2008, 2009
# 2019 -> 2020, 2021
shock_window_size = 2

# Status-to-final-value mapping
STATUS_VALUE_MAP = {
    "no_negative_growth_in_shock_window": 0,
    "not_recovered_by_available_data": 20
}

# Countries with this status are removed from final summary
REMOVE_STATUS = "no_shock_window_data"

# Output files
summary_output_file = output_dir / "country_recovery_summary_dynamic_baseline_2007_2019.csv"
audit_output_file = output_dir / "country_recovery_audit_dynamic_baseline.csv"
index_path_output_file = output_dir / "country_recovery_index_paths_dynamic_baseline.csv"
status_summary_output_file = output_dir / "country_recovery_status_summary_dynamic_baseline.csv"

In [2]:
# ------------------------------------------------------
# LOAD GDP GROWTH DATA
# ------------------------------------------------------

gdp = pd.read_csv(input_file)

required_cols = [country_col, year_col, value_col]

missing_cols = [col for col in required_cols if col not in gdp.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in GDP file: {missing_cols}")

# Clean types
gdp[country_col] = gdp[country_col].astype(str).str.strip()
gdp[year_col] = pd.to_numeric(gdp[year_col], errors="coerce")
gdp[value_col] = pd.to_numeric(gdp[value_col], errors="coerce")

# Drop rows without country/year
gdp = gdp.dropna(subset=[country_col, year_col]).copy()
gdp[year_col] = gdp[year_col].astype(int)

# Sort
gdp = gdp.sort_values([country_col, year_col]).reset_index(drop=True)

print("GDP growth data loaded.")
print("Shape:", gdp.shape)
print("Countries:", gdp[country_col].nunique())
print("Year range:", gdp[year_col].min(), "-", gdp[year_col].max())

display(gdp.head())

GDP growth data loaded.
Shape: (1314, 3)
Countries: 52
Year range: 2000 - 2025


,Country,Year,Value
0,ARG,2000,-0.788999
1,ARG,2001,-4.408840
2,ARG,2002,-10.894485
3,ARG,2003,8.837041
4,ARG,2004,9.029573


In [3]:
# ------------------------------------------------------
# DUPLICATE COUNTRY-YEAR CHECK
# ------------------------------------------------------

duplicates = (
    gdp
    .groupby([country_col, year_col])
    .size()
    .reset_index(name="row_count")
    .query("row_count > 1")
)

print("Duplicate country-year rows:", len(duplicates))

if not duplicates.empty:
    display(duplicates.head(30))

    # Collapse duplicates by mean growth value for safety.
    # If you want stricter handling, replace this with an error.
    gdp = (
        gdp
        .groupby([country_col, year_col], as_index=False)
        .agg({value_col: "mean"})
    )

    print("Duplicates collapsed by mean.")

print("Final shape after duplicate handling:", gdp.shape)

Duplicate country-year rows: 0
Final shape after duplicate handling: (1314, 3)


In [4]:
# ------------------------------------------------------
# DYNAMIC-BASELINE RECOVERY FUNCTION
# ------------------------------------------------------

def calculate_dynamic_recovery_for_country(country_df, shock_anchor_year):
    """
    Calculates recovery time using dynamic baseline logic.

    Logic:
    1. Treat shock_anchor_year as a shock-window anchor, not as fixed baseline.
       Example: 2007 or 2019.

    2. Look forward two years:
       2007 -> 2008 and 2009
       2019 -> 2020 and 2021

    3. Find the first year in this window with negative GDP growth.

    4. Dynamic baseline = year before first negative growth year.

       Example:
       2008 negative -> baseline = 2007
       2009 negative -> baseline = 2008

    5. Set GDP index in the dynamic baseline year = 100.

    6. Compound GDP growth from the first negative year onward.

    7. Recovery year = first year where cumulative GDP index returns to or exceeds 100.

    8. Recovery time = recovery_year - dynamic_baseline_year.

    Special statuses:
    - no_negative_growth_in_shock_window -> final value 0
    - not_recovered_by_available_data -> final value 20
    - no_shock_window_data -> country removed from final summary
    """

    country_df = country_df.sort_values(year_col).copy()
    country = country_df[country_col].iloc[0]

    # Dictionary: year -> GDP growth %
    growth_by_year = (
        country_df
        .dropna(subset=[value_col])
        .set_index(year_col)[value_col]
        .to_dict()
    )

    shock_window_years = [
        shock_anchor_year + i
        for i in range(1, shock_window_size + 1)
    ]

    available_window_years = [
        y for y in shock_window_years
        if y in growth_by_year
    ]

    missing_window_years = [
        y for y in shock_window_years
        if y not in growth_by_year
    ]

    # If neither of the two shock-window years is available, remove country later
    if len(available_window_years) == 0:
        result = {
            "Country": country,
            "shock_anchor_year": shock_anchor_year,
            "shock_window_years": ", ".join(map(str, shock_window_years)),
            "available_window_years": "",
            "missing_window_years": ", ".join(map(str, missing_window_years)),
            "first_negative_growth_year": np.nan,
            "dynamic_baseline_year": np.nan,
            "recovery_year": np.nan,
            "recovery_time_years": np.nan,
            "final_recovery_value": np.nan,
            "status": "no_shock_window_data",
            "min_index_after_baseline": np.nan,
            "final_index_available": np.nan,
            "final_year_available": np.nan
        }

        return result, []

    # Find first negative growth year in the shock window
    first_negative_growth_year = None

    for y in shock_window_years:
        if y in growth_by_year and growth_by_year[y] < 0:
            first_negative_growth_year = y
            break

    # If no negative growth was detected in available shock-window years
    if first_negative_growth_year is None:
        result = {
            "Country": country,
            "shock_anchor_year": shock_anchor_year,
            "shock_window_years": ", ".join(map(str, shock_window_years)),
            "available_window_years": ", ".join(map(str, available_window_years)),
            "missing_window_years": ", ".join(map(str, missing_window_years)),
            "first_negative_growth_year": np.nan,
            "dynamic_baseline_year": np.nan,
            "recovery_year": np.nan,
            "recovery_time_years": np.nan,
            "final_recovery_value": STATUS_VALUE_MAP["no_negative_growth_in_shock_window"],
            "status": "no_negative_growth_in_shock_window",
            "min_index_after_baseline": np.nan,
            "final_index_available": np.nan,
            "final_year_available": np.nan
        }

        return result, []

    # Dynamic baseline is the year before the first negative growth year
    dynamic_baseline_year = first_negative_growth_year - 1

    # Compound from the first negative year onward
    post = country_df[country_df[year_col] > dynamic_baseline_year].copy()
    post = post.dropna(subset=[value_col]).copy()

    if post.empty:
        # This should be rare because first_negative_growth_year exists,
        # but keep it for safety.
        result = {
            "Country": country,
            "shock_anchor_year": shock_anchor_year,
            "shock_window_years": ", ".join(map(str, shock_window_years)),
            "available_window_years": ", ".join(map(str, available_window_years)),
            "missing_window_years": ", ".join(map(str, missing_window_years)),
            "first_negative_growth_year": first_negative_growth_year,
            "dynamic_baseline_year": dynamic_baseline_year,
            "recovery_year": np.nan,
            "recovery_time_years": np.nan,
            "final_recovery_value": np.nan,
            "status": "no_post_baseline_data",
            "min_index_after_baseline": np.nan,
            "final_index_available": np.nan,
            "final_year_available": np.nan
        }

        return result, []

    # Build cumulative GDP index
    # Dynamic baseline year index = 100
    current_index = 100.0
    index_rows = []

    for _, row in post.iterrows():
        year = int(row[year_col])
        growth_pct = row[value_col]

        current_index = current_index * (1 + growth_pct / 100)

        index_rows.append({
            "Country": country,
            "shock_anchor_year": shock_anchor_year,
            "shock_window_years": ", ".join(map(str, shock_window_years)),
            "first_negative_growth_year": first_negative_growth_year,
            "dynamic_baseline_year": dynamic_baseline_year,
            "year": year,
            "growth_pct": growth_pct,
            "gdp_index": current_index
        })

    index_df = pd.DataFrame(index_rows)

    min_index = index_df["gdp_index"].min()
    final_index = index_df["gdp_index"].iloc[-1]
    final_year = int(index_df["year"].iloc[-1])

    # Find first year where GDP index returns to or exceeds 100
    recovery_candidates = index_df[index_df["gdp_index"] >= 100]

    if not recovery_candidates.empty:
        recovery_year = int(recovery_candidates.iloc[0]["year"])
        recovery_time_years = recovery_year - dynamic_baseline_year

        result = {
            "Country": country,
            "shock_anchor_year": shock_anchor_year,
            "shock_window_years": ", ".join(map(str, shock_window_years)),
            "available_window_years": ", ".join(map(str, available_window_years)),
            "missing_window_years": ", ".join(map(str, missing_window_years)),
            "first_negative_growth_year": first_negative_growth_year,
            "dynamic_baseline_year": dynamic_baseline_year,
            "recovery_year": recovery_year,
            "recovery_time_years": recovery_time_years,
            "final_recovery_value": recovery_time_years,
            "status": "recovered",
            "min_index_after_baseline": min_index,
            "final_index_available": final_index,
            "final_year_available": final_year
        }

        return result, index_rows

    # If GDP index never returns to/exceeds 100
    result = {
        "Country": country,
        "shock_anchor_year": shock_anchor_year,
        "shock_window_years": ", ".join(map(str, shock_window_years)),
        "available_window_years": ", ".join(map(str, available_window_years)),
        "missing_window_years": ", ".join(map(str, missing_window_years)),
        "first_negative_growth_year": first_negative_growth_year,
        "dynamic_baseline_year": dynamic_baseline_year,
        "recovery_year": np.nan,
        "recovery_time_years": np.nan,
        "final_recovery_value": STATUS_VALUE_MAP["not_recovered_by_available_data"],
        "status": "not_recovered_by_available_data",
        "min_index_after_baseline": min_index,
        "final_index_available": final_index,
        "final_year_available": final_year
    }

    return result, index_rows

In [5]:
# ------------------------------------------------------
# RUN DYNAMIC-BASELINE RECOVERY CALCULATIONS
# ------------------------------------------------------

recovery_rows = []
index_path_rows = []

for country, country_df in gdp.groupby(country_col):
    for shock_anchor_year in shock_anchor_years:
        recovery_result, index_path = calculate_dynamic_recovery_for_country(
            country_df=country_df,
            shock_anchor_year=shock_anchor_year
        )

        recovery_rows.append(recovery_result)
        index_path_rows.extend(index_path)

recovery_audit = pd.DataFrame(recovery_rows)
recovery_index_paths = pd.DataFrame(index_path_rows)

print("Recovery audit shape:", recovery_audit.shape)
print("Recovery index paths shape:", recovery_index_paths.shape)

display(recovery_audit.head(20))
display(recovery_index_paths.head(20))

Recovery audit shape: (104, 14)
Recovery index paths shape: (1020, 8)


,Country,shock_anchor_year,shock_window_years,available_window_years,missing_window_years,first_negative_growth_year,dynamic_baseline_year,recovery_year,recovery_time_years,final_recovery_value,status,min_index_after_baseline,final_index_available,final_year_available
0,ARG,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,94.081475,114.197153,2025.0
1,ARG,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2022.0,3.0,3.0,recovered,90.099515,106.611565,2025.0
2,AUS,2007,"2008, 2009","2008, 2009",,NaN,NaN,NaN,NaN,0.0,no_negative_growth_in_shock_window,NaN,NaN,NaN
3,AUS,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2021.0,2.0,2.0,recovered,97.896824,113.103204,2025.0
4,AUT,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2011.0,3.0,3.0,recovered,96.413735,115.743902,2025.0
5,AUT,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2022.0,3.0,3.0,recovered,93.681745,102.671079,2025.0
6,BEL,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,98.093582,124.658156,2025.0
7,BEL,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2021.0,2.0,2.0,recovered,95.207037,109.111504,2025.0
8,BRA,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,99.874188,130.808223,2025.0
9,BRA,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2021.0,2.0,2.0,recovered,96.723241,114.003051,2025.0


,Country,shock_anchor_year,shock_window_years,first_negative_growth_year,dynamic_baseline_year,year,growth_pct,gdp_index
0,ARG,2007,"2008, 2009",2009,2008,2009,-5.918525,94.081475
1,ARG,2007,"2008, 2009",2009,2008,2010,10.125398,103.607599
2,ARG,2007,"2008, 2009",2009,2008,2011,6.003952,109.828149
3,ARG,2007,"2008, 2009",2009,2008,2012,-1.026420,108.700850
4,ARG,2007,"2008, 2009",2009,2008,2013,2.405324,111.315458
5,ARG,2007,"2008, 2009",2009,2008,2014,-2.512615,108.518529
6,ARG,2007,"2008, 2009",2009,2008,2015,2.731160,111.482343
7,ARG,2007,"2008, 2009",2009,2008,2016,-2.080328,109.163145
8,ARG,2007,"2008, 2009",2009,2008,2017,2.818503,112.239911
9,ARG,2007,"2008, 2009",2009,2008,2018,-2.617396,109.302148


In [6]:
# ------------------------------------------------------
# REMOVE COUNTRIES WITH NO SHOCK-WINDOW DATA
# ------------------------------------------------------

countries_to_remove = (
    recovery_audit
    .query("status == @REMOVE_STATUS")["Country"]
    .unique()
    .tolist()
)

print("Countries removed due to no shock-window data:", len(countries_to_remove))
print(countries_to_remove)

recovery_audit_filtered = recovery_audit[
    ~recovery_audit["Country"].isin(countries_to_remove)
].copy()

print("Audit rows before removal:", len(recovery_audit))
print("Audit rows after removal:", len(recovery_audit_filtered))

Countries removed due to no shock-window data: 1
['SAU']
Audit rows before removal: 104
Audit rows after removal: 102


In [7]:
# ------------------------------------------------------
# CREATE FINAL COUNTRY-LEVEL SUMMARY
# ------------------------------------------------------

recovery_summary = (
    recovery_audit_filtered
    .pivot(
        index="Country",
        columns="shock_anchor_year",
        values="final_recovery_value"
    )
    .reset_index()
)

# Rename columns
recovery_summary = recovery_summary.rename(columns={
    2007: "Recovery_2007",
    2019: "Recovery_2019"
})

# Ensure both final columns exist
for col in ["Recovery_2007", "Recovery_2019"]:
    if col not in recovery_summary.columns:
        recovery_summary[col] = np.nan

# Keep only final requested columns
recovery_summary = recovery_summary[
    ["Country", "Recovery_2007", "Recovery_2019"]
].sort_values("Country").reset_index(drop=True)

# Convert to nullable integer where possible.
# This keeps clean integer values and preserves missing if any remain.
for col in ["Recovery_2007", "Recovery_2019"]:
    recovery_summary[col] = recovery_summary[col].astype("Int64")

display(recovery_summary.head(30))

shock_anchor_year,Country,Recovery_2007,Recovery_2019
0,ARG,2,3
1,AUS,0,2
2,AUT,3,3
3,BEL,2,2
4,BRA,2,2
5,CAN,2,2
6,CHE,2,2
7,CHL,2,2
8,CHN,0,0
9,COL,0,2


In [8]:
# ------------------------------------------------------
# CREATE EXPANDED AUDIT SUMMARY
# ------------------------------------------------------

expanded_summary = recovery_audit_filtered.copy()

expanded_summary = expanded_summary.sort_values(
    ["Country", "shock_anchor_year"]
).reset_index(drop=True)

display(expanded_summary.head(30))

,Country,shock_anchor_year,shock_window_years,available_window_years,missing_window_years,first_negative_growth_year,dynamic_baseline_year,recovery_year,recovery_time_years,final_recovery_value,status,min_index_after_baseline,final_index_available,final_year_available
0,ARG,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,94.081475,114.197153,2025.0
1,ARG,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2022.0,3.0,3.0,recovered,90.099515,106.611565,2025.0
2,AUS,2007,"2008, 2009","2008, 2009",,NaN,NaN,NaN,NaN,0.0,no_negative_growth_in_shock_window,NaN,NaN,NaN
3,AUS,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2021.0,2.0,2.0,recovered,97.896824,113.103204,2025.0
4,AUT,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2011.0,3.0,3.0,recovered,96.413735,115.743902,2025.0
5,AUT,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2022.0,3.0,3.0,recovered,93.681745,102.671079,2025.0
6,BEL,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,98.093582,124.658156,2025.0
7,BEL,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2021.0,2.0,2.0,recovered,95.207037,109.111504,2025.0
8,BRA,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,99.874188,130.808223,2025.0
9,BRA,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2021.0,2.0,2.0,recovered,96.723241,114.003051,2025.0


In [9]:
# ------------------------------------------------------
# QUICK COUNTRY CHECK
# ------------------------------------------------------

country_to_check = "ARG"

print("Original GDP growth rows:")
display(
    gdp[gdp[country_col] == country_to_check]
    .sort_values(year_col)
)

print("\nRecovery audit:")
display(
    recovery_audit[recovery_audit["Country"] == country_to_check]
)

print("\nFiltered recovery audit:")
display(
    recovery_audit_filtered[recovery_audit_filtered["Country"] == country_to_check]
)

print("\nGDP index path:")
display(
    recovery_index_paths[recovery_index_paths["Country"] == country_to_check]
    .sort_values(["shock_anchor_year", "year"])
)

Original GDP growth rows:


,Country,Year,Value
0,ARG,2000,-0.788999
1,ARG,2001,-4.408840
2,ARG,2002,-10.894485
3,ARG,2003,8.837041
4,ARG,2004,9.029573
5,ARG,2005,8.851660
6,ARG,2006,8.047151
7,ARG,2007,9.007651
8,ARG,2008,4.057233
9,ARG,2009,-5.918525



Recovery audit:


,Country,shock_anchor_year,shock_window_years,available_window_years,missing_window_years,first_negative_growth_year,dynamic_baseline_year,recovery_year,recovery_time_years,final_recovery_value,status,min_index_after_baseline,final_index_available,final_year_available
0,ARG,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,94.081475,114.197153,2025.0
1,ARG,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2022.0,3.0,3.0,recovered,90.099515,106.611565,2025.0



Filtered recovery audit:


,Country,shock_anchor_year,shock_window_years,available_window_years,missing_window_years,first_negative_growth_year,dynamic_baseline_year,recovery_year,recovery_time_years,final_recovery_value,status,min_index_after_baseline,final_index_available,final_year_available
0,ARG,2007,"2008, 2009","2008, 2009",,2009.0,2008.0,2010.0,2.0,2.0,recovered,94.081475,114.197153,2025.0
1,ARG,2019,"2020, 2021","2020, 2021",,2020.0,2019.0,2022.0,3.0,3.0,recovered,90.099515,106.611565,2025.0



GDP index path:


,Country,shock_anchor_year,shock_window_years,first_negative_growth_year,dynamic_baseline_year,year,growth_pct,gdp_index
0,ARG,2007,"2008, 2009",2009,2008,2009,-5.918525,94.081475
1,ARG,2007,"2008, 2009",2009,2008,2010,10.125398,103.607599
2,ARG,2007,"2008, 2009",2009,2008,2011,6.003952,109.828149
3,ARG,2007,"2008, 2009",2009,2008,2012,-1.026420,108.700850
4,ARG,2007,"2008, 2009",2009,2008,2013,2.405324,111.315458
5,ARG,2007,"2008, 2009",2009,2008,2014,-2.512615,108.518529
6,ARG,2007,"2008, 2009",2009,2008,2015,2.731160,111.482343
7,ARG,2007,"2008, 2009",2009,2008,2016,-2.080328,109.163145
8,ARG,2007,"2008, 2009",2009,2008,2017,2.818503,112.239911
9,ARG,2007,"2008, 2009",2009,2008,2018,-2.617396,109.302148


In [10]:
# ------------------------------------------------------
# STATUS SUMMARY
# ------------------------------------------------------

status_summary = (
    recovery_audit
    .groupby(["shock_anchor_year", "status"])
    .size()
    .reset_index(name="country_count")
    .sort_values(["shock_anchor_year", "status"])
)

display(status_summary)

status_summary.to_csv(status_summary_output_file, index=False)

print("Status summary exported to:")
print(status_summary_output_file)

,shock_anchor_year,status,country_count
0,2007,no_negative_growth_in_shock_window,8
1,2007,no_shock_window_data,1
2,2007,not_recovered_by_available_data,1
3,2007,recovered,42
4,2019,no_negative_growth_in_shock_window,4
5,2019,not_recovered_by_available_data,1
6,2019,recovered,47


Status summary exported to:
Data\Processed\Recovery\country_recovery_status_summary_dynamic_baseline.csv


In [12]:
# ------------------------------------------------------
# EXPORT OUTPUTS
# ------------------------------------------------------

recovery_summary.to_csv(summary_output_file, index=False)
recovery_audit.to_csv(audit_output_file, index=False)

if not recovery_index_paths.empty:
    recovery_index_paths.to_csv(index_path_output_file, index=False)

print("Final country-level recovery summary exported to:")
print(summary_output_file)

print("\nAudit details exported to:")
print(audit_output_file)

if not recovery_index_paths.empty:
    print("\nGDP index paths exported to:")
    print(index_path_output_file)

display(recovery_summary.head(30))

Final country-level recovery summary exported to:
Data\Processed\Recovery\country_recovery_summary_dynamic_baseline_2007_2019.csv

Audit details exported to:
Data\Processed\Recovery\country_recovery_audit_dynamic_baseline.csv

GDP index paths exported to:
Data\Processed\Recovery\country_recovery_index_paths_dynamic_baseline.csv


shock_anchor_year,Country,Recovery_2007,Recovery_2019
0,ARG,2,3
1,AUS,0,2
2,AUT,3,3
3,BEL,2,2
4,BRA,2,2
5,CAN,2,2
6,CHE,2,2
7,CHL,2,2
8,CHN,0,0
9,COL,0,2
